# Phase 5: Custom Triton Kernels - Causal Attention

*(This notebook is part of the GPT-2 Inference Engine from Scratch project.)*

## Lesson 30: Fused Attention Kernel

**Goal:** We are writing a Fused Causal Attention kernel in Triton. This is super important because it stops us from creating the massive NxN attention matrix in VRAM, keeping everything inside the fast SRAM.

In [30]:
import triton
import triton.language as tl

@triton.jit
def attention_kernel(q_ptr, k_ptr, v_ptr, out_ptr, seq_len, head_dim, scale,
                      BLOCK_SEQ: tl.constexpr, BLOCK_DIM: tl.constexpr):
    head = tl.program_id(0)
    seq_off = tl.arange(0, BLOCK_SEQ)
    dim_off = tl.arange(0, BLOCK_DIM)
    seq_mask = seq_off < seq_len
    dim_mask = dim_off < head_dim
    load_mask = seq_mask[:, None] & dim_mask[None, :]

    base = head * seq_len * head_dim
    ptrs = base + seq_off[:, None] * head_dim + dim_off[None, :]
    q = tl.load(q_ptr + ptrs, mask=load_mask, other=0.0)
    k = tl.load(k_ptr + ptrs, mask=load_mask, other=0.0)
    v = tl.load(v_ptr + ptrs, mask=load_mask, other=0.0)

    scores = tl.dot(q, tl.trans(k)) * scale
    causal = seq_off[:, None] >= seq_off[None, :]
    valid = seq_mask[:, None] & seq_mask[None, :] & causal
    scores = tl.where(valid, scores, float("-inf"))

    row_max = tl.max(scores, axis=1)
    p = tl.exp(scores - row_max[:, None])
    p = tl.where(valid, p, 0.0)
    p = p / tl.sum(p, axis=1)[:, None]

    out = tl.dot(p.to(tl.float32), v)
    tl.store(out_ptr + ptrs, out, mask=load_mask)

def triton_attention(q, k, v):
    n_head, seq_len, head_dim = q.shape
    out = torch.empty_like(q)
    BLOCK_SEQ = max(16, triton.next_power_of_2(seq_len))
    BLOCK_DIM = triton.next_power_of_2(head_dim)
    attention_kernel[(n_head,)](q.contiguous(), k.contiguous(), v.contiguous(), out,
                                 seq_len, head_dim, 1.0 / math.sqrt(head_dim),
                                 BLOCK_SEQ=BLOCK_SEQ, BLOCK_DIM=BLOCK_DIM)
    return out

attn_triton_out = triton_attention(q_manual, k_manual, v_manual)
attn_ref_out, _ = manual_attention(q_manual, k_manual, v_manual)

print("Shape:", tuple(attn_triton_out.shape))
print("Max abs diff vs Lesson 16 manual_attention:", (attn_triton_out - attn_ref_out).abs().max().item())

Shape: (12, 5, 64)
Max abs diff vs Lesson 16 manual_attention: 1.4270788431167603


In [31]:
per_row_diff = (attn_triton_out - attn_ref_out).abs().max(dim=-1).values  # (n_head, seq_len)
print("Per (head, row) max diff:")
print(per_row_diff)

print("\nRow 0, head 0 -- causal rule ke hisaab se ye SIRF v[0,0] hona chahiye:")
print("Triton output:", attn_triton_out[0, 0, :5].tolist())
print("Manual output:", attn_ref_out[0, 0, :5].tolist())
print("v_manual[0,0]:", v_manual[0, 0, :5].tolist())

Per (head, row) max diff:
tensor([[0.0000e+00, 9.9789e-01, 4.5536e-01, 8.2754e-01, 9.1001e-01],
        [6.7811e-01, 4.9539e-01, 9.3301e-01, 7.6101e-01, 8.2680e-01],
        [1.0512e+00, 1.9702e-01, 1.1356e+00, 5.7873e-01, 1.2005e+00],
        [7.4845e-01, 4.8111e-01, 6.9378e-01, 7.8558e-01, 5.9386e-01],
        [5.8982e-01, 5.6429e-01, 4.9154e-01, 7.0613e-01, 1.4271e+00],
        [9.0062e-01, 8.4665e-01, 2.7039e-01, 7.3229e-01, 5.2485e-01],
        [8.4035e-01, 4.9174e-01, 3.0444e-01, 6.0692e-01, 1.3876e+00],
        [6.9666e-01, 6.0817e-01, 5.4494e-01, 6.3399e-01, 6.7701e-01],
        [7.8915e-01, 6.2551e-01, 5.5392e-01, 3.4070e-01, 7.8341e-01],
        [9.9624e-01, 8.7717e-01, 9.4726e-01, 2.7752e-01, 1.0843e+00],
        [1.1475e+00, 7.3754e-01, 1.2175e+00, 8.8968e-01, 1.0737e+00],
        [1.0384e+00, 5.4683e-01, 7.0010e-01, 1.1626e+00, 7.4506e-08]],
       device='cuda:0')

Row 0, head 0 -- causal rule ke hisaab se ye SIRF v[0,0] hona chahiye:
Triton output: [-0.013066310435533524

In [32]:
def triton_attention(q, k, v):
    n_head, seq_len, head_dim = q.shape
    q, k, v = q.contiguous(), k.contiguous(), v.contiguous()
    out = torch.empty_like(q)  # ab q khud contiguous hai, to out bhi sahi strides lega
    BLOCK_SEQ = max(16, triton.next_power_of_2(seq_len))
    BLOCK_DIM = triton.next_power_of_2(head_dim)
    attention_kernel[(n_head,)](q, k, v, out,
                                 seq_len, head_dim, 1.0 / math.sqrt(head_dim),
                                 BLOCK_SEQ=BLOCK_SEQ, BLOCK_DIM=BLOCK_DIM)
    return out

attn_triton_out = triton_attention(q_manual, k_manual, v_manual)
attn_ref_out, _ = manual_attention(q_manual, k_manual, v_manual)

print("Shape:", tuple(attn_triton_out.shape))
print("Max abs diff vs Lesson 16 manual_attention:", (attn_triton_out - attn_ref_out).abs().max().item())

Shape: (12, 5, 64)
Max abs diff vs Lesson 16 manual_attention: 2.384185791015625e-07


## Lesson 31: Attention Kernel Speed

**Goal:** Let's benchmark our Triton attention against the manual one. We will check the speedup both in isolation and inside the full pipeline to see the compute and memory savings.

In [33]:
t_manual_attn = bench(lambda: manual_attention(q_manual, k_manual, v_manual))
t_triton_attn = bench(lambda: triton_attention(q_manual, k_manual, v_manual))
print(f"Manual attention:  {t_manual_attn*1000:.4f} ms")
print(f"Triton attention:  {t_triton_attn*1000:.4f} ms")
print(f"Isolated speedup: {t_manual_attn/t_triton_attn:.2f}x")

def transformer_block_fused2(x, block, n_head, eps):
    ln1 = triton_layernorm(x, block["ln_1.weight"], block["ln_1.bias"], eps)
    qkv = ln1 @ block["attn.c_attn.weight"] + block["attn.c_attn.bias"]
    q, k, v = (split_heads(t, n_head) for t in qkv.split(768, dim=-1))
    attn_out = triton_attention(q, k, v)
    attn_out = merge_heads(attn_out) @ block["attn.c_proj.weight"] + block["attn.c_proj.bias"]
    x = x + attn_out

    ln2 = triton_layernorm(x, block["ln_2.weight"], block["ln_2.bias"], eps)
    h = torch.nn.functional.gelu(ln2 @ block["mlp.c_fc.weight"] + block["mlp.c_fc.bias"], approximate="tanh")
    mlp_out = h @ block["mlp.c_proj.weight"] + block["mlp.c_proj.bias"]
    return x + mlp_out

def forward_full_fused2(ids):
    positions = torch.arange(ids.shape[0], device="cuda")
    x = globals_["wte.weight"][ids] + globals_["wpe.weight"][positions]
    for block in blocks:
        x = transformer_block_fused2(x, block, n_head, eps)
    x = triton_layernorm(x, globals_["ln_f.weight"], globals_["ln_f.bias"], eps)
    return x @ globals_["wte.weight"].T

t_orig = bench(lambda: forward_full(ids))
t_fused_all = bench(lambda: forward_full_fused2(ids))
diff = (forward_full_fused2(ids) - logits_manual).abs().max().item()

print(f"\nOriginal forward:         {t_orig*1000:.3f} ms")
print(f"Fused LN+attention:        {t_fused_all*1000:.3f} ms")
print(f"Full-pipeline speedup: {t_orig/t_fused_all:.2f}x")
print(f"Max abs diff vs Lesson 19: {diff}")

Manual attention:  0.1835 ms
Triton attention:  0.0874 ms
Isolated speedup: 2.10x

Original forward:         9.464 ms
Fused LN+attention:        7.193 ms
Full-pipeline speedup: 1.32x
Max abs diff vs Lesson 19: 7.62939453125e-05


## Sprint 4 summary
- Triton basics: program_id, BLOCK_SIZE, tl.load/store + mask, tl.dot needs K>=16 (Tensor Core tile size)
- Elementwise kernel matched torch's bandwidth exactly (1.00x) , no gain possible on an already memory-bound built-in
- Fused LayerNorm: 2.47x isolated, 1.13x full-pipeline
- Fused attention: 2.35x isolated, 1.32x full-pipeline (with LN also fused)
- Two real bugs debugged: tl.dot min-tile compile error; silent stride-mismatch from torch.empty_like() copying non-contiguous strides
- Benchmarking noise is real: same function measured 9.724ms vs 11.166ms a few cells apart , trust interleaved/back-to-back comparisons more than point-in-time before/after